# Phase 1 — SBERT Text Representation Experiment

This notebook evaluates **SBERT** as the text representation for Software Requirement Prioritization using **LightGBM Ranker** as the fixed baseline ranking model.

**Objective:** Determine whether SBERT produces a strong feature representation for requirement prioritization.

**Model:** `all-MiniLM-L6-v2` → 384-dimensional embeddings (pre-computed in master dataset)

**Ranker:** LightGBM (default parameters, no hyperparameter tuning)


In [1]:
import logging
import os
import json
import joblib
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score, average_precision_score
import lightgbm as lgb
import mlflow

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("Phase1_SBERT_Text_Representation")
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


In [2]:
# Configuration
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384  # Expected SBERT dimension
RANDOM_STATE = 42
TEST_SIZE = 0.3
OUTPUT_DIR = "outputs/phase1_sbert"
DATASET_PATH = "dataset/sentence_embedding.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)
logger.info(f"Output directory: {OUTPUT_DIR}")


2026-07-19 21:55:51,772 - INFO - Output directory: outputs/phase1_sbert


## STEP 1 — Load Dataset

Load the master dataset from `dataset/sentence_embedding.csv`. This is the source of truth containing all original metadata and pre-computed SBERT embeddings.


In [3]:
df = pd.read_csv(DATASET_PATH)
logger.info(f"Dataset shape: {df.shape}")
logger.info(f"Columns: {list(df.columns)}")

# Detect requirement text column
text_keywords = ["requirement", "text", "description", "sentence", "story", "content"]
text_cols = [c for c in df.columns if any(k in c.lower() for k in text_keywords)]
if text_cols:
    logger.info(f"Detected requirement text column(s): {text_cols}")
else:
    logger.info("No explicit requirement text column found. Using pre-computed embeddings.")

# Validate required columns
required_cols = {"id", "project_id", "type", "value", "effort", "risk", "stakeholder_priority", "rank"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

embed_cols = [c for c in df.columns if c.startswith("embedding_")]
if not embed_cols:
    raise ValueError("No embedding columns found in dataset. "
                     "Expected pre-computed SBERT embeddings (embedding_0..embedding_N)")

logger.info(f"Found {len(embed_cols)} embedding columns ({embed_cols[0]} to {embed_cols[-1]})")
logger.info("All required columns present.")

display(df.head(3))
display(df.describe())


2026-07-19 21:55:51,819 - INFO - Dataset shape: (698, 256)
2026-07-19 21:55:51,820 - INFO - Columns: ['id', 'project_id', 'type', 'value', 'effort', 'risk', 'stakeholder_priority', 'priority_score', 'rank', 'embedding_0', 'embedding_1', 'embedding_2', 'embedding_3', 'embedding_4', 'embedding_5', 'embedding_6', 'embedding_7', 'embedding_8', 'embedding_9', 'embedding_10', 'embedding_11', 'embedding_12', 'embedding_13', 'embedding_14', 'embedding_15', 'embedding_16', 'embedding_17', 'embedding_18', 'embedding_19', 'embedding_20', 'embedding_21', 'embedding_22', 'embedding_23', 'embedding_24', 'embedding_25', 'embedding_26', 'embedding_27', 'embedding_28', 'embedding_29', 'embedding_30', 'embedding_31', 'embedding_32', 'embedding_33', 'embedding_34', 'embedding_35', 'embedding_36', 'embedding_37', 'embedding_38', 'embedding_39', 'embedding_40', 'embedding_41', 'embedding_42', 'embedding_43', 'embedding_44', 'embedding_45', 'embedding_46', 'embedding_47', 'embedding_48', 'embedding_49', 'em

,id,project_id,type,value,effort,risk,stakeholder_priority,priority_score,rank,embedding_0,...,embedding_237,embedding_238,embedding_239,embedding_240,embedding_241,embedding_242,embedding_243,embedding_244,embedding_245,embedding_246
0,REQ-01,P1,FR,3,2,1,3,1.3,18,0.030996,...,-0.020718,0.002305,-0.015420,-0.077480,-0.009985,-0.070100,0.017221,0.060295,0.020246,0.008327
1,REQ-02,P1,FR,3,2,1,3,1.3,18,-0.008261,...,-0.013755,0.024546,0.010715,0.008381,-0.095685,-0.141046,-0.076318,-0.037939,0.025249,0.093777
2,REQ-03,P1,FR,3,2,1,3,1.3,18,-0.032415,...,-0.023209,0.017611,0.017243,-0.032235,-0.094481,-0.107027,-0.075131,0.001994,0.029203,0.068259


,value,effort,risk,stakeholder_priority,priority_score,rank,embedding_0,embedding_1,embedding_2,embedding_3,...,embedding_237,embedding_238,embedding_239,embedding_240,embedding_241,embedding_242,embedding_243,embedding_244,embedding_245,embedding_246
count,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,...,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000
mean,3.656160,2.859599,2.651862,3.653295,1.424642,60.270774,-0.019273,0.035761,-0.015619,-0.011795,...,-0.007449,-0.017867,0.023431,-0.019016,-0.004069,-0.013686,0.019134,0.001424,0.027367,-0.000537
std,0.773928,0.847175,1.186003,0.778198,0.417646,58.346216,0.047024,0.049242,0.044376,0.043276,...,0.037911,0.048960,0.049613,0.057588,0.049573,0.047710,0.043370,0.048459,0.047810,0.050980
min,2.000000,1.000000,1.000000,2.000000,0.200000,1.000000,-0.139028,-0.139118,-0.144867,-0.132398,...,-0.132605,-0.159222,-0.120924,-0.169999,-0.153380,-0.155949,-0.103009,-0.169575,-0.140185,-0.146081
25%,3.000000,2.000000,2.000000,3.000000,1.100000,11.000000,-0.051346,0.001004,-0.044163,-0.040164,...,-0.034274,-0.048899,-0.008384,-0.058988,-0.035859,-0.044420,-0.010732,-0.030434,-0.006422,-0.033640
50%,4.000000,3.000000,2.000000,4.000000,1.400000,39.000000,-0.022426,0.037553,-0.016763,-0.011617,...,-0.007251,-0.020916,0.022473,-0.019424,-0.006151,-0.010858,0.020260,0.000026,0.024328,0.000509
75%,4.000000,3.000000,3.000000,4.000000,1.700000,101.000000,0.013443,0.070997,0.013969,0.016815,...,0.019024,0.010462,0.056104,0.020157,0.026223,0.018978,0.049306,0.031424,0.060829,0.032526
max,5.000000,5.000000,5.000000,5.000000,2.700000,232.000000,0.130875,0.214671,0.137820,0.132734,...,0.130366,0.138529,0.187646,0.144586,0.140415,0.119563,0.139498,0.174564,0.234068,0.164476


## STEP 2 — Data Validation

Perform validation checks: missing values, duplicates, invalid project IDs, invalid rank values, and data types.


In [4]:
validation_report = {}

# Missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
validation_report["missing_values"] = len(missing_cols)
if len(missing_cols) > 0:
    logger.warning(f"Columns with missing values:\n{missing_cols}")
else:
    logger.info("No missing values found.")

# Duplicate rows
dup_rows = df.duplicated().sum()
validation_report["duplicate_rows"] = dup_rows
if dup_rows > 0:
    logger.warning(f"Duplicate rows: {dup_rows}")
else:
    logger.info("No duplicate rows found.")

# Duplicate requirements (by id)
dup_ids = df["id"].duplicated().sum()
validation_report["duplicate_ids"] = dup_ids
if dup_ids > 0:
    logger.warning(f"Duplicate requirement IDs: {dup_ids}")

# Invalid project_id
invalid_pid = df["project_id"].isnull().sum() | (df["project_id"].astype(str).str.strip() == "").sum()
validation_report["invalid_project_ids"] = int(invalid_pid)
if invalid_pid > 0:
    logger.warning(f"Invalid project IDs: {invalid_pid}")

# Invalid rank (should be numeric, non-negative)
invalid_rank = (~pd.to_numeric(df["rank"], errors="coerce").notna()).sum()
validation_report["invalid_rank"] = int(invalid_rank)
if invalid_rank > 0:
    logger.warning(f"Invalid rank values: {invalid_rank}")

# Data types
validation_report["dtypes"] = {c: str(dt) for c, dt in df.dtypes.items()}

logger.info("=== Validation Report ===")
for k, v in validation_report.items():
    logger.info(f"  {k}: {v}")


2026-07-19 21:55:52,088 - INFO - No missing values found.
2026-07-19 21:55:52,105 - INFO - No duplicate rows found.
2026-07-19 21:55:52,107 - INFO - === Validation Report ===
2026-07-19 21:55:52,107 - INFO -   missing_values: 0
2026-07-19 21:55:52,109 - INFO -   duplicate_rows: 0
2026-07-19 21:55:52,109 - INFO -   duplicate_ids: 0
2026-07-19 21:55:52,109 - INFO -   invalid_project_ids: 0
2026-07-19 21:55:52,109 - INFO -   invalid_rank: 0
2026-07-19 21:55:52,111 - INFO -   dtypes: {'id': 'object', 'project_id': 'object', 'type': 'object', 'value': 'int64', 'effort': 'int64', 'risk': 'int64', 'stakeholder_priority': 'int64', 'priority_score': 'float64', 'rank': 'int64', 'embedding_0': 'float64', 'embedding_1': 'float64', 'embedding_2': 'float64', 'embedding_3': 'float64', 'embedding_4': 'float64', 'embedding_5': 'float64', 'embedding_6': 'float64', 'embedding_7': 'float64', 'embedding_8': 'float64', 'embedding_9': 'float64', 'embedding_10': 'float64', 'embedding_11': 'float64', 'embeddin

## STEP 3 — SBERT Text Representation

The master dataset contains pre-computed SBERT embeddings generated using `all-MiniLM-L6-v2`. These are 384-dimensional sentence embeddings stored in columns `embedding_0` through `embedding_383`.

**Note:** The actual embedding dimension in the dataset is determined dynamically.


In [5]:
embed_cols = sorted([c for c in df.columns if c.startswith("embedding_")],
                      key=lambda x: int(x.split("_")[1]))

actual_dim = len(embed_cols)
logger.info(f"Embedding dimension: {actual_dim}")

embedding_matrix = df[embed_cols].values
logger.info(f"Embedding matrix shape: {embedding_matrix.shape}")

logger.info(f"First embedding vector (first 10 dims): {embedding_matrix[0, :10]}")
logger.info(f"First embedding vector shape: {embedding_matrix[0].shape}")

# Quick distribution check
logger.info(f"Embedding value range: [{embedding_matrix.min():.4f}, {embedding_matrix.max():.4f}]")
logger.info(f"Mean: {embedding_matrix.mean():.6f}, Std: {embedding_matrix.std():.6f}")


2026-07-19 21:55:52,117 - INFO - Embedding dimension: 247
2026-07-19 21:55:52,120 - INFO - Embedding matrix shape: (698, 247)
2026-07-19 21:55:52,120 - INFO - First embedding vector (first 10 dims): [ 0.03099552  0.08201721 -0.03240236  0.04377936  0.02469208 -0.00162476
 -0.02168863 -0.03060964  0.06035456  0.02331645]
2026-07-19 21:55:52,120 - INFO - First embedding vector shape: (247,)
2026-07-19 21:55:52,121 - INFO - Embedding value range: [-0.2546, 0.2341]
2026-07-19 21:55:52,126 - INFO - Mean: 0.000907, Std: 0.050827


## STEP 4 — Feature Fusion

Create the final feature matrix by combining:
- **Numerical features:** `value`, `effort`, `risk`, `stakeholder_priority`
- **Text features:** `embedding_0` ... `embedding_N`

**Excluded:**
- `id`, `priority_score` (target leakage), `rank` (target), `project_id` (query group)


In [6]:
num_features = ["value", "effort", "risk", "stakeholder_priority"]
feature_cols = num_features + embed_cols

logger.info(f"Numerical features: {num_features}")
logger.info(f"Total feature dimension: {len(feature_cols)} (4 numerical + {len(embed_cols)} embedding)")

X = df[feature_cols].copy()
logger.info(f"Feature matrix shape: {X.shape}")
display(X.head(3))


2026-07-19 21:55:52,136 - INFO - Numerical features: ['value', 'effort', 'risk', 'stakeholder_priority']
2026-07-19 21:55:52,137 - INFO - Total feature dimension: 251 (4 numerical + 247 embedding)
2026-07-19 21:55:52,141 - INFO - Feature matrix shape: (698, 251)


,value,effort,risk,stakeholder_priority,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_237,embedding_238,embedding_239,embedding_240,embedding_241,embedding_242,embedding_243,embedding_244,embedding_245,embedding_246
0,3,2,1,3,0.030996,0.082017,-0.032402,0.043779,0.024692,-0.001625,...,-0.020718,0.002305,-0.015420,-0.077480,-0.009985,-0.070100,0.017221,0.060295,0.020246,0.008327
1,3,2,1,3,-0.008261,0.051770,-0.009457,0.019407,-0.009472,0.069842,...,-0.013755,0.024546,0.010715,0.008381,-0.095685,-0.141046,-0.076318,-0.037939,0.025249,0.093777
2,3,2,1,3,-0.032415,0.011653,-0.014935,0.021126,0.021682,0.049995,...,-0.023209,0.017611,0.017243,-0.032235,-0.094481,-0.107027,-0.075131,0.001994,0.029203,0.068259


## STEP 5 — Feature Encoding

One-hot encode the `type` column (FR/NFR) and append to the feature matrix.


In [7]:
encoded_type = pd.get_dummies(df["type"], prefix="type")
logger.info(f"One-hot encoded type columns: {list(encoded_type.columns)}")

X = pd.concat([X, encoded_type], axis=1)
logger.info(f"Feature matrix shape after encoding: {X.shape}")

display(X.head(3))


2026-07-19 21:55:52,155 - INFO - One-hot encoded type columns: ['type_FR', 'type_NFR']
2026-07-19 21:55:52,157 - INFO - Feature matrix shape after encoding: (698, 253)


,value,effort,risk,stakeholder_priority,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_239,embedding_240,embedding_241,embedding_242,embedding_243,embedding_244,embedding_245,embedding_246,type_FR,type_NFR
0,3,2,1,3,0.030996,0.082017,-0.032402,0.043779,0.024692,-0.001625,...,-0.015420,-0.077480,-0.009985,-0.070100,0.017221,0.060295,0.020246,0.008327,True,False
1,3,2,1,3,-0.008261,0.051770,-0.009457,0.019407,-0.009472,0.069842,...,0.010715,0.008381,-0.095685,-0.141046,-0.076318,-0.037939,0.025249,0.093777,True,False
2,3,2,1,3,-0.032415,0.011653,-0.014935,0.021126,0.021682,0.049995,...,0.017243,-0.032235,-0.094481,-0.107027,-0.075131,0.001994,0.029203,0.068259,True,False


## STEP 6 — Query Group Construction

Prepare the dataset for Learning to Rank:
- **Query/group identifier:** `project_id`
- **Ranking label:** `rank`
- Verify every project contains multiple requirements.


In [8]:
# Transform rank into lambdarank-compatible labels
# Within each group: dense rank (0 = worst, N-1 = best)
# Lower original rank = higher priority → higher label
df["label"] = df.groupby("project_id")["rank"].transform(
    lambda x: x.rank(method="dense", ascending=False).astype(int) - 1
)

y = df["label"].values
groups = df["project_id"].values
query_ids = df["project_id"]

group_sizes = query_ids.value_counts()
logger.info(f"Number of query groups (projects): {len(group_sizes)}")
logger.info(f"Group size stats:\n{group_sizes.describe()}")

single_req_groups = (group_sizes == 1).sum()
if single_req_groups > 0:
    logger.warning(f"Groups with only 1 requirement: {single_req_groups} — these cannot be ranked")

# Compute group counts for LightGBM
group_counts = group_sizes.sort_index().values  # sorted by project_id
group_order = group_sizes.index.sort_values()
logger.info(f"Group sizes: {group_counts[:10]}... (showing first 10)")

# Validate label distribution
logger.info(f"Label value range: [{y.min()}, {y.max()}]")
logger.info(f"Unique labels: {sorted(np.unique(y))[:20]}...")


2026-07-19 21:55:52,178 - INFO - Number of query groups (projects): 15
2026-07-19 21:55:52,179 - INFO - Group size stats:
count     15.000000
mean      46.533333
std       64.790725
min        3.000000
25%       10.000000
50%       18.000000
75%       49.500000
max      231.000000
Name: count, dtype: float64
2026-07-19 21:55:52,182 - INFO - Group sizes: [ 47   8  18  10  19   9   3 231  99 146]... (showing first 10)
2026-07-19 21:55:52,182 - INFO - Label value range: [0, 20]
2026-07-19 21:55:52,182 - INFO - Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]...


## STEP 7 — Train/Test Split

Use `GroupShuffleSplit` to ensure the same project never appears in both train and test sets. Fixed `random_state=42` for reproducibility.


In [9]:
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test = y[test_idx]
groups_train = groups[train_idx]
groups_test = groups[test_idx]

# Recompute group counts for train/test
train_group_counts = pd.Series(groups_train).value_counts().sort_index().values
test_group_counts = pd.Series(groups_test).value_counts().sort_index().values

logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups: {len(np.unique(groups_train))}, Test groups: {len(np.unique(groups_test))}")

# Verify no overlap
train_projects = set(np.unique(groups_train))
test_projects = set(np.unique(groups_test))
overlap = train_projects & test_projects
if overlap:
    raise ValueError(f"Project overlap between train and test: {overlap}")
logger.info("No project overlap between train and test sets. Split is clean.")


2026-07-19 21:55:52,211 - INFO - Train size: 434, Test size: 264
2026-07-19 21:55:52,212 - INFO - Train groups: 10, Test groups: 5
2026-07-19 21:55:52,213 - INFO - No project overlap between train and test sets. Split is clean.


## STEP 8 — LightGBM Ranker Training

Train a baseline LightGBM Ranker with reasonable default parameters. No hyperparameter tuning — the goal is to evaluate SBERT representation quality, not to optimize the ranker.


In [10]:
max_group_size = df.groupby("project_id").size().max()
num_leaves = min(max_group_size, 255)

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    boosting_type="gbdt",
    n_estimators=100,
    num_leaves=num_leaves,
    learning_rate=0.1,
    min_child_samples=10,
    random_state=RANDOM_STATE,
    verbose=-1
)

logger.info("Training LightGBM Ranker...")
ranker.fit(
    X_train, y_train,
    group=train_group_counts,
    eval_set=[(X_test, y_test)],
    eval_group=[test_group_counts],
    eval_metric=["ndcg"],
    callbacks=[lgb.log_evaluation(0)]
)
logger.info("Training complete.")

logger.info(f"Feature importances (top 10):\n"
            f"{pd.Series(ranker.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)}")


2026-07-19 21:55:52,229 - INFO - Training LightGBM Ranker...
2026-07-19 21:55:54,278 - INFO - Training complete.
2026-07-19 21:55:54,279 - INFO - Feature importances (top 10):
effort           96
value            66
embedding_94     47
embedding_233    41
embedding_106    40
embedding_120    39
embedding_43     34
embedding_169    34
embedding_231    33
embedding_70     32
dtype: int32


## STEP 9 — Model Evaluation

Evaluate using:
- **NDCG@5, NDCG@10** — Normalized Discounted Cumulative Gain
- **MAP** — Mean Average Precision
- **Spearman Rank Correlation**
- **Kendall Tau**


In [11]:
y_pred = ranker.predict(X_test)

# Compute NDCG per query group, then average
test_project_ids = groups_test
ndcg5_scores = []
ndcg10_scores = []
map_per_group = []

for pid in np.unique(test_project_ids):
    mask = test_project_ids == pid
    y_true_group = y_test[mask]
    y_pred_group = y_pred[mask]
    n = len(y_true_group)

    # NDCG
    if n >= 5:
        k5 = min(5, n)
        ndcg5_scores.append(ndcg_score(y_true_group.reshape(1, -1),
                                        y_pred_group.reshape(1, -1), k=k5))
    if n >= 10:
        k10 = min(10, n)
        ndcg10_scores.append(ndcg_score(y_true_group.reshape(1, -1),
                                         y_pred_group.reshape(1, -1), k=k10))

    # MAP (binarize: top half of labels = relevant)
    if len(np.unique(y_true_group)) > 1:
        threshold = y_true_group.max() * 0.5
        y_bin = (y_true_group >= threshold).astype(int)
        if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
            map_per_group.append(average_precision_score(y_bin, y_pred_group))

ndcg5 = float(np.mean(ndcg5_scores)) if ndcg5_scores else 0.0
ndcg10 = float(np.mean(ndcg10_scores)) if ndcg10_scores else 0.0
map_score = float(np.mean(map_per_group)) if map_per_group else 0.0

spearman_corr, spearman_p = spearmanr(y_test, y_pred)
kendall_corr, kendall_p = kendalltau(y_test, y_pred)

metrics = {
    "NDCG_at_5": round(float(ndcg5), 6),
    "NDCG_at_10": round(float(ndcg10), 6),
    "MAP": round(float(map_score), 6),
    "Spearman": round(float(spearman_corr), 6),
    "Spearman_pvalue": round(float(spearman_p), 6),
    "KendallTau": round(float(kendall_corr), 6),
    "KendallTau_pvalue": round(float(kendall_p), 6)
}

logger.info("=== Evaluation Metrics ===")
for k, v in metrics.items():
    logger.info(f"  {k}: {v}")

display(pd.DataFrame([metrics]))


2026-07-19 21:55:54,307 - INFO - === Evaluation Metrics ===
2026-07-19 21:55:54,309 - INFO -   NDCG_at_5: 0.687576
2026-07-19 21:55:54,310 - INFO -   NDCG_at_10: 0.751521
2026-07-19 21:55:54,310 - INFO -   MAP: 0.751672
2026-07-19 21:55:54,311 - INFO -   Spearman: 0.518793
2026-07-19 21:55:54,311 - INFO -   Spearman_pvalue: 0.0
2026-07-19 21:55:54,312 - INFO -   KendallTau: 0.370093
2026-07-19 21:55:54,313 - INFO -   KendallTau_pvalue: 0.0


,NDCG_at_5,NDCG_at_10,MAP,Spearman,Spearman_pvalue,KendallTau,KendallTau_pvalue
0,0.687576,0.751521,0.751672,0.518793,0.0,0.370093,0.0


## STEP 10 — Save Outputs

Save all experiment outputs to `outputs/phase1_sbert/`:
- Trained LightGBM model
- Generated SBERT dataset (for Phase 2 input)
- Evaluation metrics (JSON)
- Predictions
- Feature list


In [12]:
# 1. Save trained model
model_path = os.path.join(OUTPUT_DIR, "lgbm_ranker.pkl")
joblib.dump(ranker, model_path)
logger.info(f"Model saved: {model_path}")

# 2. Save generated dataset (Phase 2 input)
dataset_out = X.copy()
dataset_out["label"] = y
dataset_out["rank"] = df["rank"].values
dataset_out["project_id"] = groups
dataset_path = os.path.join(OUTPUT_DIR, "sbert_dataset.csv")
dataset_out.to_csv(dataset_path, index=False)
logger.info(f"Generated dataset saved: {dataset_path} (shape: {dataset_out.shape})")

# 3. Save evaluation metrics
metrics_path = os.path.join(OUTPUT_DIR, "evaluation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
logger.info(f"Metrics saved: {metrics_path}")

# 4. Save predictions
predictions_df = pd.DataFrame({
    "project_id": groups_test,
    "true_label": y_test,
    "predicted_score": y_pred
})
pred_path = os.path.join(OUTPUT_DIR, "predictions.csv")
predictions_df.to_csv(pred_path, index=False)
logger.info(f"Predictions saved: {pred_path}")

# 5. Save feature list
features_path = os.path.join(OUTPUT_DIR, "feature_list.txt")
with open(features_path, "w") as f:
    f.write("\n".join(X.columns.tolist()))
logger.info(f"Feature list saved: {features_path}")


2026-07-19 21:55:54,344 - INFO - Model saved: outputs/phase1_sbert\lgbm_ranker.pkl
2026-07-19 21:55:54,500 - INFO - Generated dataset saved: outputs/phase1_sbert\sbert_dataset.csv (shape: (698, 256))
2026-07-19 21:55:54,500 - INFO - Metrics saved: outputs/phase1_sbert\evaluation_metrics.json
2026-07-19 21:55:54,504 - INFO - Predictions saved: outputs/phase1_sbert\predictions.csv
2026-07-19 21:55:54,505 - INFO - Feature list saved: outputs/phase1_sbert\feature_list.txt


## STEP 11 — MLflow Logging

Track the experiment using MLflow: log parameters, metrics, and artifacts.


In [13]:
with mlflow.start_run(run_name="SBERT_all-MiniLM-L6-v2_LightGBM") as run:
    # Log parameters
    mlflow.log_param("embedding_model", EMBEDDING_MODEL_NAME)
    mlflow.log_param("embedding_dimension", actual_dim)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("num_train_groups", len(np.unique(groups_train)))
    mlflow.log_param("num_test_groups", len(np.unique(groups_test)))
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("test_size_ratio", TEST_SIZE)
    mlflow.log_param("num_features", X.shape[1])
    mlflow.log_param("ranker_objective", "lambdarank")
    mlflow.log_param("ranker_n_estimators", 100)
    mlflow.log_param("ranker_num_leaves", num_leaves)

    # Log metrics
    mlflow.log_metrics(metrics)

    # Log artifacts
    mlflow.log_artifact(model_path, artifact_path="model")
    mlflow.log_artifact(metrics_path, artifact_path="metrics")
    mlflow.log_artifact(features_path, artifact_path="features")
    mlflow.log_artifact(dataset_path, artifact_path="dataset")
    mlflow.log_artifact(pred_path, artifact_path="predictions")

    logger.info(f"MLflow run ID: {run.info.run_id}")
    logger.info(f"MLflow experiment logged successfully.")


2026-07-19 21:55:55,015 - INFO - MLflow run ID: 1b5084324d1443f98448ff81967481a3
2026-07-19 21:55:55,016 - INFO - MLflow experiment logged successfully.


## Experiment Summary

### What was done
1. **Dataset:** Loaded master dataset with pre-computed SBERT embeddings
2. **Features:** 4 numerical features + embedding dimensions + one-hot encoded type
3. **Model:** LightGBM Ranker (Lambdarank objective, default parameters)
4. **Split:** GroupShuffleSplit (30% test, no project overlap)
5. **Evaluation:** NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau


In [14]:
summary_data = {
    "Metric": ["NDCG@5", "NDCG@10", "MAP", "Spearman rho", "Kendall tau"],
    "Value": [
        metrics["NDCG_at_5"],
        metrics["NDCG_at_10"],
        metrics["MAP"],
        metrics["Spearman"],
        metrics["KendallTau"]
    ]
}
summary_df = pd.DataFrame(summary_data)

logger.info("=== Experiment Summary ===")
logger.info(f"Embedding model: {EMBEDDING_MODEL_NAME}")
logger.info(f"Embedding dimension: {actual_dim}")
logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups: {len(np.unique(groups_train))}, Test groups: {len(np.unique(groups_test))}")
logger.info(f"Total features: {X.shape[1]}")
logger.info(f"\n{summary_df.to_string(index=False)}")

display(summary_df)


2026-07-19 21:55:55,042 - INFO - === Experiment Summary ===
2026-07-19 21:55:55,043 - INFO - Embedding model: all-MiniLM-L6-v2
2026-07-19 21:55:55,045 - INFO - Embedding dimension: 247
2026-07-19 21:55:55,046 - INFO - Train size: 434, Test size: 264
2026-07-19 21:55:55,047 - INFO - Train groups: 10, Test groups: 5
2026-07-19 21:55:55,048 - INFO - Total features: 253
2026-07-19 21:55:55,050 - INFO - 
      Metric    Value
      NDCG@5 0.687576
     NDCG@10 0.751521
         MAP 0.751672
Spearman rho 0.518793
 Kendall tau 0.370093


,Metric,Value
0,NDCG@5,0.687576
1,NDCG@10,0.751521
2,MAP,0.751672
3,Spearman rho,0.518793
4,Kendall tau,0.370093


### Outputs
All artifacts saved to `outputs/phase1_sbert/`. The generated dataset is ready for Phase 2 (Ranking Algorithm Experiment).

---
*Phase 1 — SBERT Text Representation Experiment complete.*